In [1]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                              bidirectional=True, divide_output=True, pscan=True, use_cuda=False)
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [3]:
class ROIPatchEmbed3D(nn.Module):
    def __init__(self, n_rois=2, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois = n_rois
        self.patch_size = patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, rois):
        batch_size, n_rois = rois.shape[:2]
        x = rois.reshape(batch_size * n_rois, 1, rois.shape[-3], rois.shape[-2], rois.shape[-1])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(batch_size, n_rois, self.patches_per_roi, self.d_model)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, None, :, :] + self.roi_embed.weight[None, :, None, :]
        occupancy = F.max_pool3d((x.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool().reshape(batch_size, n_rois, self.patches_per_roi)
        tokens = tokens.reshape(batch_size, -1, self.d_model)
        valid = valid.reshape(batch_size, -1)
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class VisionMambaBranch(nn.Module):
    def __init__(self, n_rois=2, roi_size=64, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)
        return pooled


class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=2, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        pooled = self.branch(rois)
        return self.classifier(self.dropout(pooled))

In [4]:
COHORT_CSV     = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_roi64_custompad_aug"  # <- new cache
CKPT_DIR       = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
if df["subject_id"].nunique() != len(df):
    raise ValueError("Split must be subject-level.")

sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# left+right hemisphere pairs -- indices match the (6, 64, 64, 64) cache layout
ROI_GROUPS = {
    'Hippocampus': [0, 1],
    'Cerebellum-WM': [2, 3],
    'Cerebral-WM': [4, 5],
}

Train: 126 | Val: 42 | Test: 42


In [5]:
class PairedROIDataset(Dataset):
    """Slices the region-appropriate-padding 6-ROI cache down to ONE
    left+right pair (2 channels), given as a list of indices."""
    def __init__(self, sessions, labels, cache_dir, roi_indices, is_train=False):
        self.samples, self.cache_dir, self.roi_indices = [], cache_dir, roi_indices
        for session_id, label in zip(sessions, labels):
            self.samples.append((session_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        session_id, label, version = self.samples[idx]
        full_rois = np.array(np.load(f"{self.cache_dir}/{session_id}_{version}.npy", mmap_mode="r"),
                              dtype=np.float32, copy=True)
        rois = full_rois[self.roi_indices]  # (2, 64, 64, 64)
        return torch.from_numpy(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), session_id

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels, _ in loader:
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(rois), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for rois, labels, _ in loader:
            rois, labels = rois.to(device), labels.to(device)
            out = model(rois)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [7]:
def measure_inference_time(model, loader, device, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            rois, labels, _ = batch
            rois = rois.to(device)
            bs = rois.shape[0]
            if device.type == 'cuda': torch.cuda.synchronize()
            t0 = time.time()
            _ = model(rois)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)

def try_compute_flops(model, loader, device):
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            rois, labels, _ = batch
            macs, _ = profile(model, inputs=(rois[:1].to(device),), verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None

def run_one_seed(seed, model_kwargs, train_loader, val_loader, test_loader, save_prefix, max_epochs=101, patience=15):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = VisionMambaModel(**model_kwargs, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = evaluate(model, test_loader, criterion, device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device)
    flops = try_compute_flops(model, test_loader, device)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops}

In [8]:
BATCH_SIZE = 4
PAIR_SEEDS = [1, 7, 123]

paired_roi_custompad_results = {}

for group_name, indices in ROI_GROUPS.items():
    print(f"\n{'='*20} ROI pair: {group_name} {indices} {'='*20}")

    train_loader = DataLoader(PairedROIDataset(X_train, y_train, MRI_CACHE_AUG, indices, True), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(PairedROIDataset(X_val, y_val, MRI_CACHE_AUG, indices, False), batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(PairedROIDataset(X_test, y_test, MRI_CACHE_AUG, indices, False), batch_size=BATCH_SIZE, shuffle=False)

    seed_results = [
        run_one_seed(s, {"n_rois": 2}, train_loader, val_loader, test_loader,
                     save_prefix=f"vim_pair_custompad_mri_{group_name.replace('-', '_')}")
        for s in PAIR_SEEDS
    ]
    paired_roi_custompad_results[group_name] = seed_results

    accs = [r["acc"] for r in seed_results]
    tprs = [r["tpr"] for r in seed_results]
    tnrs = [r["tnr"] for r in seed_results]
    print(f"\n{group_name} (3-seed): Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
          f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
          f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}%")


==================== ROI pair: Hippocampus [0, 1] ====================

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7027 |     0.6924 |   0.5952 |   0.3810 |   0.8095 |   9.3s
     2 |     0.6910 |     0.6909 |   0.5238 |   0.1905 |   0.8571 |   9.0s
     3 |     0.6824 |     0.6894 |   0.5714 |   0.3333 |   0.8095 |   9.0s
     4 |     0.6729 |     0.6893 |   0.5476 |   0.5714 |   0.5238 |   8.9s
     5 |     0.6691 |     0.6917 |   0.5238 |   0.1905 |   0.8571 |   8.8s
     6 |     0.6657 |     0.6893 |   0.5000 |   0.4762 |   0.5238 |   9.0s
     7 |     0.6607 |     0.6887 |   0.5238 |   0.5714 |   0.4762 |   9.0s
     8 |     0.6548 |     0.6880 |   0.5000 |   0.3810 |   0.6190 |   8.8s
     9 |     0.6472 |     0.6878 |   0.4762 |   0.2857 |   0.6667 |   9.0s
    10 |     0.6334 |     0.6926 |   0.5238 |   0.1429 |   0.9048 |   8.9s
    11 |     0.6

In [9]:
print(f"{'ROI Pair':<18} | {'Accuracy':>14} | {'TPR':>14} | {'TNR':>14}")
print("-" * 65)
for group_name, seed_results in paired_roi_custompad_results.items():
    accs = [r["acc"] for r in seed_results]
    tprs = [r["tpr"] for r in seed_results]
    tnrs = [r["tnr"] for r in seed_results]
    print(f"{group_name:<18} | {np.mean(accs)*100:>6.1f}±{np.std(accs,ddof=1)*100:<5.1f}% | "
          f"{np.mean(tprs)*100:>6.1f}±{np.std(tprs,ddof=1)*100:<5.1f}% | "
          f"{np.mean(tnrs)*100:>6.1f}±{np.std(tnrs,ddof=1)*100:<5.1f}%")

print("\n=== Compare to original pad=3 paired results ===")
print("Hippocampus:    56.3±7.3% | TPR=60.3±12.0% | TNR=52.4±12.6%")
print("Cerebellum-WM:  65.1±2.7% | TPR=50.8±9.9%  | TNR=79.4±15.3%")
print("Cerebral-WM:    69.0±0.0% | TPR=61.9±8.2%  | TNR=76.2±8.2%")

ROI Pair           |       Accuracy |            TPR |            TNR
-----------------------------------------------------------------
Hippocampus        |   58.7±7.3  % |   60.3±5.5  % |   57.1±19.0 %
Cerebellum-WM      |   57.1±2.4  % |   49.2±2.7  % |   65.1±7.3  %
Cerebral-WM        |   69.0±0.0  % |   61.9±8.2  % |   76.2±8.2  %

=== Compare to original pad=3 paired results ===
Hippocampus:    56.3±7.3% | TPR=60.3±12.0% | TNR=52.4±12.6%
Cerebellum-WM:  65.1±2.7% | TPR=50.8±9.9%  | TNR=79.4±15.3%
Cerebral-WM:    69.0±0.0% | TPR=61.9±8.2%  | TNR=76.2±8.2%
